# Chapter 3 — Multiple Regression: Estimation
## University of Neuchâtel — Python for Econometrics

Topics:
1. Omitted variable bias — why simple regression misleads
2. Matrix OLS: $\hat{\beta} = (X'X)^{-1}X'y$
3. R² and adjusted R²
4. Standard errors
5. Frisch-Waugh-Lovell (FWL) theorem
6. Multicollinearity and VIF

**Format:** One cell per step. Manual first, statsmodels second, `np.isclose()` comparison.

## 0 — Setup

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import wooldridge as woo
import os

In [ ]:
chapter_dir = os.path.dirname(os.path.abspath('__file__'))
os.chdir(chapter_dir)
output_dir = os.path.join(chapter_dir, 'output')
os.makedirs(output_dir, exist_ok=True)
print('Working directory:', chapter_dir)

## 1 — Load Data

In [ ]:
df = woo.dataWoo('wage1')
print('Shape:', df.shape)
print(df[['wage','educ','exper','tenure']].head(10))

## 2 — Omitted Variable Bias

**Simple regression:** $wage = \beta_0 + \beta_1 educ + u$

If `exper` belongs in the model and $Cov(educ, exper) \neq 0$,
omitting `exper` causes **bias**:

$$bias = \tilde{\beta}_1 - \beta_1 \approx \delta_1 \cdot \frac{Cov(exper, educ)}{Var(educ)}$$

where $\delta_1$ is the effect of `exper` on `wage`.

### Step 2.1 — Simple regression: wage ~ educ (misspecified)

In [ ]:
res_simple = smf.ols('wage ~ educ', data=df).fit()
print(f'Simple: b1(educ) = {res_simple.params["educ"]:.4f}')

### Step 2.2 — Multiple regression: wage ~ educ + exper

In [ ]:
res_multi = smf.ols('wage ~ educ + exper', data=df).fit()
print(f'Multiple: b1(educ) = {res_multi.params["educ"]:.4f}')
bias_est = res_simple.params['educ'] - res_multi.params['educ']
print(f'Estimated omitted variable bias: {bias_est:.4f}')
print('The bias is positive because educ and exper are negatively correlated')
print(f'Corr(educ, exper) = {np.corrcoef(df["educ"], df["exper"])[0,1]:.4f}')

## 3 — Matrix OLS: $\hat{\beta} = (X'X)^{-1}X'y$

**Full model:** $wage = \beta_0 + \beta_1 educ + \beta_2 exper + \beta_3 tenure + u$

In matrix form: $\mathbf{y} = \mathbf{X}\boldsymbol{\beta} + \mathbf{u}$

OLS minimises $\mathbf{u}'\mathbf{u}$, yielding: $\hat{\boldsymbol{\beta}} = (\mathbf{X}'\mathbf{X})^{-1}\mathbf{X}'\mathbf{y}$

### Step 3.1 — Build design matrix X

In [ ]:
n = len(df)
X = np.column_stack([
    np.ones(n),
    df['educ'].values,
    df['exper'].values,
    df['tenure'].values
])
y = df['wage'].values
print('X shape:', X.shape)
print('First 5 rows of X (const, educ, exper, tenure):')
print(X[:5])

### Step 3.2 — Compute X'X

In [ ]:
XtX = X.T @ X
print('X prime X (4x4 matrix):')
print(XtX.round(2))

### Step 3.3 — Compute X'y

In [ ]:
Xty = X.T @ y
print('X prime y:')
print(Xty.round(4))

### Step 3.4 — Solve for beta_hat

In [ ]:
beta_hat = np.linalg.solve(XtX, Xty)
labels = ['const','educ','exper','tenure']
print('Manual OLS beta_hat:')
for lbl, b in zip(labels, beta_hat):
    print(f'  {lbl:8s}: {b:.4f}')

### Step 3.5 — Statsmodels OLS

In [ ]:
res = smf.ols('wage ~ educ + exper + tenure', data=df).fit()
print(res.summary())

### Step 3.6 — Compare manual vs statsmodels

In [ ]:
sm_p = res.params
print('Coefficient comparison:')
for lbl, b_man in zip(labels, beta_hat):
    match = np.isclose(b_man, sm_p[lbl])
    print(f'  {lbl:8s}: manual={b_man:.4f}, sm={sm_p[lbl]:.4f}, match={match}')

## 4 — Fitted Values, Residuals, R²

### Step 4.1 — Fitted values and residuals manually

In [ ]:
y_hat = X @ beta_hat
e     = y - y_hat
print('First 5 fitted values and residuals:')
for i in range(5):
    print(f'  y={y[i]:.2f}, y_hat={y_hat[i]:.2f}, e={e[i]:.2f}')

### Step 4.2 — SST, SSR, SSE, R² manually

In [ ]:
y_mean = y.mean()
SST = np.sum((y - y_mean)**2)
SSR = np.sum(e**2)
SSE = SST - SSR
R2_manual = 1 - SSR / SST
print(f'SST = {SST:.4f}')
print(f'SSR = {SSR:.4f}')
print(f'SSE = {SSE:.4f}')
print(f'R2  = {R2_manual:.4f}')

In [ ]:
print(f'R2 match: {np.isclose(R2_manual, res.rsquared)}')
print(f'SSR match: {np.isclose(SSR, res.ssr)}')

### Step 4.3 — Adjusted R² manually

In [ ]:
k = X.shape[1] - 1  # number of regressors (excl. constant)
adjR2_manual = 1 - (SSR / (n - k - 1)) / (SST / (n - 1))
print(f'Adjusted R² (manual): {adjR2_manual:.4f}')
print(f'Adjusted R² (sm):     {res.rsquared_adj:.4f}')
print(f'Match: {np.isclose(adjR2_manual, res.rsquared_adj)}')

## 5 — Standard Errors

$$\hat{\sigma}^2 = \frac{SSR}{n - k - 1}$$

$$\widehat{Var}(\hat{\boldsymbol{\beta}}) = \hat{\sigma}^2 (X'X)^{-1}$$

$$SE(\hat{\beta}_j) = \sqrt{\hat{\sigma}^2 [(X'X)^{-1}]_{jj}}$$

### Step 5.1 — sigma_hat² manually

In [ ]:
sigma2_hat = SSR / (n - k - 1)
sigma_hat  = np.sqrt(sigma2_hat)
print(f'sigma_hat²: {sigma2_hat:.4f}')
print(f'sigma_hat:  {sigma_hat:.4f}')
print(f'Match with sm: {np.isclose(sigma_hat, res.mse_resid**0.5)}')

### Step 5.2 — Covariance matrix of beta_hat

In [ ]:
V_hat = sigma2_hat * np.linalg.inv(XtX)
SE_manual = np.sqrt(np.diag(V_hat))
print('Standard errors (manual):')
for lbl, se in zip(labels, SE_manual):
    print(f'  {lbl:8s}: {se:.4f}')

In [ ]:
print('SE comparison:')
for lbl, se_man in zip(labels, SE_manual):
    se_sm = res.bse[lbl]
    match = np.isclose(se_man, se_sm)
    print(f'  {lbl:8s}: manual={se_man:.4f}, sm={se_sm:.4f}, match={match}')

## 6 — Frisch-Waugh-Lovell Theorem

**FWL:** The coefficient on $educ$ in the full model equals the OLS slope from
regressing the residuals of $wage$ on $(exper, tenure)$ on the residuals of
$educ$ on $(exper, tenure)$.

This shows that multiple regression "controls for" the other variables.

### Step 6.1 — Partial out exper and tenure from wage

In [ ]:
res_y_on_xz = smf.ols('wage ~ exper + tenure', data=df).fit()
y_tilde = res_y_on_xz.resid.values
print('Residuals of wage ~ exper + tenure:')
print(y_tilde[:5].round(4))

### Step 6.2 — Partial out exper and tenure from educ

In [ ]:
res_x_on_xz = smf.ols('educ ~ exper + tenure', data=df).fit()
x_tilde = res_x_on_xz.resid.values
print('Residuals of educ ~ exper + tenure:')
print(x_tilde[:5].round(4))

### Step 6.3 — OLS of y_tilde ~ x_tilde

In [ ]:
b1_fwl = np.sum(x_tilde * y_tilde) / np.sum(x_tilde**2)
print(f'FWL slope on educ: {b1_fwl:.4f}')
print(f'Full model b1:     {beta_hat[1]:.4f}')
print(f'Match: {np.isclose(b1_fwl, beta_hat[1])}')
print('Intuition: FWL isolates the pure effect of educ, net of exper and tenure')

## 7 — Multicollinearity and VIF

**Variance Inflation Factor (VIF)** for variable $j$:

$$VIF_j = \frac{1}{1 - R^2_j}$$

where $R^2_j$ is from regressing $x_j$ on all other regressors.

Rule of thumb: $VIF > 10$ signals severe multicollinearity.

In [ ]:
print('Correlation matrix of regressors:')
print(df[['educ','exper','tenure']].corr().round(3))

In [ ]:
for var, others in [('educ','exper + tenure'),('exper','educ + tenure'),('tenure','educ + exper')]:
    r2_j = smf.ols(f'{var} ~ {others}', data=df).fit().rsquared
    vif  = 1 / (1 - r2_j)
    print(f'VIF({var}) = {vif:.3f}  (R2={r2_j:.4f})')

## 8 — Summary Table

In [ ]:
sumtab = pd.DataFrame({
    'Variable': labels,
    'Manual_b': beta_hat,
    'SM_b':     [res.params[l] for l in labels],
    'Manual_SE': SE_manual,
    'SM_SE':    [res.bse[l] for l in labels],
    'Match_b':  [np.isclose(beta_hat[i], res.params[labels[i]]) for i in range(4)]
})
print(sumtab.round(4).to_string(index=False))
sumtab.to_csv(os.path.join(output_dir,'table_ch03_ols_comparison.csv'), index=False)
print('Saved table_ch03_ols_comparison.csv')